In [ ]:
# Load required packages
if (!require("fs")) install.packages("fs")  # For better file system operations
library(fs)

# Define paths using forward slashes for R
train_path <- "content/train"
test_path <- "content/test"

# Function to delete directory if it exists
delete_if_exists <- function(path) {
  if (dir_exists(path)) {
    dir_delete(path)
    cat("Deleted:", path, "\n")
  } else {
    cat("Folder does not exist:", path, "\n")
  }
}

# Delete train and test folders
folders <- c(train_path, test_path)
lapply(folders, delete_if_exists)

Deleted: /content/train
Deleted: /content/test


In [ ]:
# Install and load required packages
if (!require("magick")) install.packages("magick")
library(magick)

# Define source paths
dataset_root <- "content/Dataset"  # Adjust based on your actual path
train_path <- file.path(dataset_root, "train")
test_path <- file.path(dataset_root, "test")

# Define target paths for saving
output_folders <- list(
  train_sketch = file.path("content", "train", "sketch"),
  train_real = file.path("content", "train", "real"),
  test_sketch = file.path("content", "test", "sketch"),
  test_real = file.path("content", "test", "real")
)

# Create output directories if they don't exist
lapply(output_folders, dir.create, showWarnings = FALSE, recursive = TRUE)

# Function to split and save images
process_images <- function(input_folder, sketch_folder, real_folder) {
  # Get list of jpg files
  image_files <- list.files(
    path = input_folder, 
    pattern = "\\.jpe?g$", 
    full.names = TRUE
  )
  
  for (img_path in image_files) {
    tryCatch({
      # Read image
      img <- image_read(img_path)
      
      # Get image dimensions
      dims <- image_info(img)
      mid <- as.integer(dims$width / 2)
      
      # Split image into two halves
      sketch <- image_crop(img, geometry = sprintf("%dx%d+%d+0", mid, dims$height, mid))
      real_face <- image_crop(img, geometry = sprintf("%dx%d+0+0", mid, dims$height))
      
      # Get image name
      img_name <- basename(img_path)
      
      # Save images
      image_write(sketch, path = file.path(sketch_folder, img_name))
      image_write(real_face, path = file.path(real_folder, img_name))
      
      cat("Processed:", img_name, "\n")
    }, error = function(e) {
      cat("Failed to process", img_path, ":", e$message, "\n")
    })
  }
}

# Process Train & Test Data
process_images(train_path, output_folders$train_sketch, output_folders$train_real)
process_images(test_path, output_folders$test_sketch, output_folders$test_real)

cat("\nSeparation Complete!\n")

Processed: 20024.jpg
Processed: 3494.jpg
Processed: 2702.jpg
Processed: 10242.jpg
Processed: 3550.jpg
Processed: 358.jpg
Processed: 3806.jpg
Processed: 2078.jpg
Processed: 10910.jpg
Processed: 362.jpg
Processed: 2384.jpg
Processed: 3532.jpg
Processed: 2770.jpg
Processed: 702.jpg
Processed: 8682.jpg
Processed: 7320.jpg
Processed: 20638.jpg
Processed: 3650.jpg
Processed: 2000.jpg
Processed: 740.jpg
Processed: 4740.jpg
Processed: 14966.jpg
Processed: 19436.jpg
Processed: 13876.jpg
Processed: 218.jpg
Processed: 10468.jpg
Processed: 2982.jpg
Processed: 1806.jpg
Processed: 2884.jpg
Processed: 13250.jpg
Processed: 5236.jpg
Processed: 3906.jpg
Processed: 2088.jpg
Processed: 626.jpg
Processed: 2550.jpg
Processed: 4452.jpg
Processed: 12070.jpg
Processed: 21162.jpg
Processed: 1132.jpg
Processed: 3296.jpg
Processed: 12042.jpg
Processed: 2598.jpg
Processed: 4564.jpg
Processed: 3292.jpg
Processed: 4198.jpg
Processed: 3280.jpg
Processed: 1172.jpg
Processed: 18626.jpg
Processed: 90.jpg
Processed: 7694

In [ ]:
# Install required packages
if (!require("keras")) install.packages("keras")
if (!require("tensorflow")) install.packages("tensorflow")
if (!require("e1071")) install.packages("e1071")  # For SVM
if (!require("progress")) install.packages("progress")  # For progress bar
if (!require("caret")) install.packages("caret")  # For train/test split

# Load libraries
library(keras)
library(tensorflow)
library(e1071)
library(progress)
library(caret)

# Initialize Keras and load pre-trained VGG16
vgg_model <- application_vgg16(
  weights = "imagenet",
  include_top = FALSE,
  input_shape = c(224, 224, 3)
)

# Function to extract features from an image
extract_features <- function(image_path) {
  # Read and preprocess image
  img <- image_load(image_path, target_size = c(224, 224))
  img_array <- image_to_array(img)
  img_array <- array_reshape(img_array, c(1, 224, 224, 3))
  img_array <- imagenet_preprocess_input(img_array)
  
  # Extract features
  features <- predict(vgg_model, img_array)
  return(as.vector(features))  # Flatten to 1D
}

# Set paths
sketch_folder <- file.path("content", "train", "sketch")
real_folder <- file.path("content", "train", "real")

# Initialize lists for features
X <- list()  # Sketch features
Y <- list()  # Real face features

# Get all sketch image paths
sketch_paths <- list.files(sketch_folder, pattern = "\\.jpg$", full.names = TRUE)
total_images <- length(sketch_paths)

# Create progress bar
pb <- progress_bar$new(
  format = "(:spin) [:bar] :percent | Remaining: :eta",
  total = total_images,
  clear = FALSE
)

# Extract features
for (sketch_path in sketch_paths) {
  img_name <- basename(sketch_path)
  real_path <- file.path(real_folder, img_name)
  
  if (!file.exists(real_path)) {
    cat("Skipping", img_name, ", real face not found\n")
    next
  }
  
  # Extract features
  tryCatch({
    sketch_features <- extract_features(sketch_path)
    real_features <- extract_features(real_path)
    
    X[[length(X) + 1]] <- sketch_features
    Y[[length(Y) + 1]] <- real_features
    
    pb$tick()
  }, error = function(e) {
    cat("Error processing", img_name, ":", e$message, "\n")
  })
}

# Convert lists to matrices
X <- do.call(rbind, X)
Y <- do.call(rbind, Y)

# Split data into training and testing sets
set.seed(42)
train_index <- createDataPartition(1:nrow(X), p = 0.8, list = FALSE)
X_train <- X[train_index, ]
X_test <- X[-train_index, ]
Y_train <- Y[train_index, ]
Y_test <- Y[-train_index, ]

# Train SVM model
cat("\nTraining SVM Model...\n")
svm_models <- list()
for (i in 1:ncol(Y_train)) {
  svm_models[[i]] <- svm(
    x = X_train,
    y = Y_train[, i],
    kernel = "radial",
    type = "eps-regression"
  )
  cat("Trained model for output", i, "of", ncol(Y_train), "\n")
}

cat("✅ SVM Training Complete!\n")


Extracting Features:   1%|          | 35/4739 [00:10<24:03,  3.26image/s]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step



Extracting Features:   0%|          | 1/4739 [00:00<48:49,  1.62image/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step



Extracting Features:   0%|          | 2/4739 [00:00<27:57,  2.82image/s]

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step



Extracting Features: 100%|█████████▉| 4738/4739 [20:36<00:00,  3.41image/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step



Extracting Features: 100%|██████████| 4739/4739 [20:37<00:00,  3.83image/s]



Training SVM Model...
